# Kaggle T4 Test Notebook

This notebook verifies a Kaggle T4 GPU runtime and runs small tensor benchmarks and memory checks to confirm correct GPU behavior.

In [ ]:
# Section 1: Check Kaggle GPU Runtime
import os, sys, time, subprocess
import torch
print('Python', sys.version.split('\n')[0])
print('Torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        print('GPU:', torch.cuda.get_device_name(0))
    except Exception as e:
        print('Could not get device name:', e)
print('/kaggle/input exists:', os.path.exists('/kaggle/input'))

In [ ]:
# Section 2: Import Core Libraries
import os, sys, time, subprocess, json
import torch
import numpy as np
print('Imports ready')

In [ ]:
# Section 3: Detect T4 and CUDA Availability
cuda = torch.cuda.is_available()
info = {'cuda': cuda}
if cuda:
    try:
        name = torch.cuda.get_device_name(0)
        info['name'] = name
        props = torch.cuda.get_device_properties(0)
        info['major'] = props.major
        info['minor'] = props.minor
        info['total_memory_GB'] = round(props.total_memory / (1024**3), 2)
    except Exception as e:
        info['error'] = str(e)
print(json.dumps(info, indent=2))
if not cuda:
    print('No CUDA device found; switch notebook accelerator to GPU (T4)')

In [ ]:
# Section 4: Print Device and Driver Details
try:
    print('Torch CUDA version:', torch.version.cuda)
except Exception:
    print('Torch CUDA version: unknown')
# nvidia-smi (may not be available in some restricted environments)
try:
    out = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], stderr=subprocess.STDOUT)
    print('nvidia-smi:')
    print(out.decode())
except Exception as e:
    print('nvidia-smi not available or failed:', e)
# nvcc version
try:
    out = subprocess.check_output(['nvcc', '--version'])
    print('nvcc:')
    print(out.decode().split('\n')[-2])
except Exception:
    print('nvcc not available')

In [ ]:
# Section 5: Run a Small Tensor Operation on GPU
results = {}
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    sz = 1024
    a = torch.randn((sz, sz), device=device)
    b = torch.randn((sz, sz), device=device)
    torch.cuda.synchronize()
    t0 = time.time()
    c = a @ b
    torch.cuda.synchronize()
    t1 = time.time()
    results['matmul_time_s'] = t1 - t0
    results['allocated_MB'] = torch.cuda.memory_allocated() / (1024**2)
    results['reserved_MB'] = torch.cuda.memory_reserved() / (1024**2)
    print('GPU matmul time (s):', results['matmul_time_s'])
    print('GPU allocated MB:', results['allocated_MB'])
else:
    print('Skipping GPU tensor op: no CUDA device')

In [ ]:
# Section 6: Benchmark GPU vs CPU Speed
bench = {}
iters = 5
cpu_time = 0.0
gpu_time = 0.0
# CPU
for i in range(iters):
    a = torch.randn((512,512))
    b = torch.randn((512,512))
    t0 = time.time()
    c = a @ b
    t1 = time.time()
    cpu_time += (t1 - t0)
bench['cpu_avg_s'] = cpu_time / iters
# GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.synchronize()
    for i in range(iters):
        a = torch.randn((512,512), device=device)
        b = torch.randn((512,512), device=device)
        torch.cuda.synchronize()
        t0 = time.time()
        c = a @ b
        torch.cuda.synchronize()
        t1 = time.time()
        gpu_time += (t1 - t0)
    bench['gpu_avg_s'] = gpu_time / iters

print('Benchmark:', bench)
results['benchmark'] = bench

In [ ]:
# Section 7: Inspect GPU Memory Usage
try:
    if torch.cuda.is_available():
        print('Allocated MB:', torch.cuda.memory_allocated()/(1024**2))
        print('Reserved MB:', torch.cuda.memory_reserved()/(1024**2))
        print('Max Allocated MB:', torch.cuda.max_memory_allocated()/(1024**2))
    else:
        print('No CUDA device to inspect memory')
except Exception as e:
    print('Memory inspect failed:', e)

# Run a small allocation/free cycle
if torch.cuda.is_available():
    device = torch.device('cuda')
    torch.cuda.empty_cache()
    x = torch.zeros((1024,1024), device=device)
    print('After alloc - allocated MB:', torch.cuda.memory_allocated()/(1024**2))
    del x
    torch.cuda.empty_cache()
    print('After free - allocated MB:', torch.cuda.memory_allocated()/(1024**2))

In [ ]:
# Section 8: Save Test Results to File
out_path = '/kaggle/working/kaggle_t4_test_results.json'
try:
    with open(out_path, 'w') as f:
        json.dump(results, f, indent=2)
    print('Saved results to', out_path)
except Exception as e:
    print('Failed to save results:', e)

print('\nDone.\nInstructions: Set this notebook accelerator to GPU (T4) in Notebook Settings, attach the dataset via "Add data", then run all cells.')